# Refugio Animal: gestor de animales con Tkinter

## Rol de este cuaderno

Este cuaderno es la **defensa y la documentación narrada** del ejercicio:
explica sección a sección el núcleo (diccionario + CRUD + Tkinter).

El **programa completo** vive en la carpeta `tarea1/`:

```bash
cd tarea1
python app.py
```

| Capa | Carpeta / módulo | Contenido |
| --- | --- | --- |
| Arranque | `app.py`, `main.py` | Ventana y `mainloop` |
| Datos | `datos/` | Inventario CRUD, ficha clínica, adjuntos, **modelos POO** |
| Validación | `validaciones/reglas.py` | `validar_datos` (edad 0–30, peso 0.01–100 kg) |
| Interfaz | `interfaz/` | `GestorAnimales`, ficha clínica, estilos, portapapeles |
| Exportación | `exportacion/pdf_fichas.py` | PDF de fichas |
| Tests / docs | `tests/`, `docs/` | unittest + RF/CU/UML/manuales |

Guía oral corta: `tarea1/Analisis_defensa_gestion_animales.md`.

## Qué demuestra este cuaderno vs `tarea1/`

| En este notebook (versión didáctica) | En `tarea1/` (entrega) |
| --- | --- |
| Dict + CRUD + GUI en celdas | Mismos conceptos, código modular |
| Validación de edad/peso | Igual + clínica, PDF, adjuntos ≤ 5 MB |
| Mini capa POO (herencia / `@property`) | `datos/modelos.py` + `adaptador_poo.py` |

Los datos viven **en memoria**; al reiniciar el kernel se pierden los cambios de la sesión.

## Documentación de `Tema3-InterfacesGraficas` utilizada

| Lección | Aplicación en el proyecto |
| --- | --- |
| [02 - Gestores de geometría](Tema3-InterfacesGraficas/02_GestoresGeometria.ipynb) | `pack()` en bloques; `grid()` dentro de `LabelFrame` |
| [03 - Variables de control](Tema3-InterfacesGraficas/03_VariablesControl.ipynb) | `StringVar` en el formulario |
| [04 - Frame y LabelFrame](Tema3-InterfacesGraficas/04_Contenedor_Frame_Labelframe.ipynb) | Formulario, filtros, tabla |
| [08 - Button](Tema3-InterfacesGraficas/08_Widget_Button.ipynb) / [09 - Entry](Tema3-InterfacesGraficas/09_Widget_Entry.ipynb) | Acciones CRUD y campos |
| [10 - Eventos](Tema3-InterfacesGraficas/10_EventosGenericos.ipynb) | `KeyRelease`, `<<ComboboxSelected>>`, `<<TreeviewSelect>>` |
| [11 - Cajas de diálogo](Tema3-InterfacesGraficas/11_CajasDialogo.ipynb) | `messagebox` |
| [13 - Widgets TTK](Tema3-InterfacesGraficas/13_WidgetsTematicos_TTK.ipynb) | Controles `ttk` |
| [17 - Combobox](Tema3-InterfacesGraficas/17_Widget_Combobox.ipynb) / [18 - Text](Tema3-InterfacesGraficas/18_Widget_Text.ipynb) | Especie y observaciones |
| [22 - Treeview](Tema3-InterfacesGraficas/22_Widget_Treeview.ipynb) | Tabla de inventario |

Tema 2 (POO): ver sección **2b** más abajo y `Tema2-POO/`.


## 1. Datos iniciales y estructura

`animales` es un diccionario cuyas claves son identificadores numéricos únicos
(como una clave primaria). Cada valor es otro diccionario con los campos de la ficha.

- **Edad:** entero entre 0 y 30.
- **Peso:** número finito entre `PESO_MIN_KG` (0.01) y `PESO_MAX_KG` (100).
- **Especies:** las mismas que en `tarea1/datos/constantes.py`.


In [ ]:
import math
import tkinter as tk
from abc import ABC, abstractmethod
from functools import wraps
from tkinter import messagebox, ttk


ESPECIES = ("Perro", "Gato", "Conejo", "Loro", "Hámster")
PESO_MIN_KG = 0.01
PESO_MAX_KG = 100.0

animales = {
    1: {
        "nombre": "Luna",
        "especie": "Perro",
        "edad": 4,
        "peso": 18.5,
        "color": "Canela",
        "observaciones": "Sociable; disfruta de los paseos.",
    },
    2: {
        "nombre": "Misi",
        "especie": "Gato",
        "edad": 3,
        "peso": 4.2,
        "color": "Blanco y gris",
        "observaciones": "Tranquilo; busca una familia paciente.",
    },
    3: {
        "nombre": "Kiko",
        "especie": "Loro",
        "edad": 7,
        "peso": 0.6,
        "color": "Verde",
        "observaciones": "Le gusta escuchar música.",
    },
    4: {
        "nombre": "Toby",
        "especie": "Perro",
        "edad": 1,
        "peso": 8.3,
        "color": "Negro",
        "observaciones": "Juguetón y con mucha energía.",
    },
}

print(f"Registros iniciales: {len(animales)}")
print("Especies:", ESPECIES)
print("Primer registro:", animales[1])


## 2. Operaciones CRUD y validación

Las funciones separan la **gestión de datos** de la interfaz. Reciben el
diccionario como argumento (se pueden probar sin GUI). Los errores se
comunican con `ValueError`; la interfaz los muestra con `messagebox`.

Misma idea que en `tarea1/validaciones/reglas.py` + `tarea1/datos/inventario.py`.


In [ ]:
def validar_datos(datos):
    """Comprueba los datos del formulario y devuelve un registro limpio."""
    nombre = str(datos.get("nombre", "")).strip()
    especie = str(datos.get("especie", "")).strip()
    color = str(datos.get("color", "")).strip()
    observaciones = str(datos.get("observaciones", "")).strip()
    edad = datos.get("edad")

    if not nombre:
        raise ValueError("El nombre es obligatorio.")
    if especie not in ESPECIES:
        raise ValueError("Selecciona una especie de la lista.")
    if not color:
        raise ValueError("El color es obligatorio.")
    if isinstance(edad, bool) or not isinstance(edad, int):
        raise ValueError("La edad debe ser un número entero.")
    if edad < 0 or edad > 30:
        raise ValueError("La edad debe estar entre 0 y 30 años.")

    try:
        peso = float(datos.get("peso"))
    except (TypeError, ValueError) as error:
        raise ValueError("El peso debe ser un número válido.") from error
    if not math.isfinite(peso):
        raise ValueError("El peso debe ser un número finito.")
    if peso < PESO_MIN_KG or peso > PESO_MAX_KG:
        raise ValueError(
            f"El peso debe estar entre {PESO_MIN_KG} y {PESO_MAX_KG} kg."
        )

    return {
        "nombre": nombre,
        "especie": especie,
        "edad": edad,
        "peso": peso,
        "color": color,
        "observaciones": observaciones,
    }


def crear_animal(datos, inventario):
    """Valida y añade un animal; devuelve el nuevo identificador."""
    registro = validar_datos(datos)
    nuevo_id = max(inventario, default=0) + 1
    inventario[nuevo_id] = registro
    return nuevo_id


def leer_animal(animal_id, inventario):
    """Devuelve una copia de un registro para evitar cambios accidentales."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario[animal_id].copy()


def actualizar_animal(animal_id, datos, inventario):
    """Valida y sustituye los datos del animal indicado."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    inventario[animal_id] = validar_datos(datos)


def borrar_animal(animal_id, inventario):
    """Elimina el animal indicado y devuelve sus datos."""
    if animal_id not in inventario:
        raise ValueError("No se ha encontrado ese animal.")
    return inventario.pop(animal_id)


## 2b. Capa POO (Tema 2): herencia, `@property` y decoradores

El enunciado pide **diccionarios** para el CRUD. En `tarea1/` hay además una
capa didáctica:

- `AnimalRefugio` (ABC) → `Perro`, `Gato`, `Conejo`, `Loro`, `Hamster`
- Getters/setters con `@property` / `@….setter`
- Decoradores: `@abstractmethod`, `@classmethod`, y uno propio `@cuenta_usos`
- Adaptador `dict` ↔ objeto en `datos/adaptador_poo.py`

La celda siguiente es una **versión reducida** para la defensa oral (el código
completo está en `tarea1/datos/modelos.py`).


In [ ]:
def cuenta_usos(metodo):
    """Decorador propio: cuenta invocaciones en self._usos_<nombre>."""

    @wraps(metodo)
    def envoltura(self, *args, **kwargs):
        atributo = f"_usos_{metodo.__name__}"
        setattr(self, atributo, getattr(self, atributo, 0) + 1)
        return metodo(self, *args, **kwargs)

    return envoltura


class AnimalRefugio(ABC):
    """Contrato común (no se instancia)."""

    ESPECIE = ""

    def __init__(self, nombre, edad, peso, color, observaciones=""):
        self.nombre = nombre
        self.edad = edad
        self.peso = peso
        self.color = color
        self.observaciones = observaciones

    @property
    def nombre(self):
        return self._nombre

    @nombre.setter
    def nombre(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("El nombre es obligatorio.")
        self._nombre = valor.strip()

    @property
    def edad(self):
        return self._edad

    @edad.setter
    def edad(self, valor):
        if isinstance(valor, bool) or not isinstance(valor, int):
            raise ValueError("La edad debe ser un número entero.")
        if valor < 0 or valor > 30:
            raise ValueError("La edad debe estar entre 0 y 30 años.")
        self._edad = valor

    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        peso = float(valor)
        if not math.isfinite(peso) or peso < PESO_MIN_KG or peso > PESO_MAX_KG:
            raise ValueError(
                f"El peso debe estar entre {PESO_MIN_KG} y {PESO_MAX_KG} kg."
            )
        self._peso = peso

    @property
    def color(self):
        return self._color

    @color.setter
    def color(self, valor):
        if not isinstance(valor, str) or not valor.strip():
            raise ValueError("El color es obligatorio.")
        self._color = valor.strip()

    @property
    def observaciones(self):
        return self._observaciones

    @observaciones.setter
    def observaciones(self, valor):
        self._observaciones = str(valor or "").strip()

    @property
    def especie(self):
        return self.ESPECIE

    @cuenta_usos
    def info_basica(self):
        return f"{self.nombre} ({self.especie}), {self.edad} años, {self.peso:.2f} kg"

    @abstractmethod
    def hacer_sonido(self):
        ...

    @abstractmethod
    def tipo_alimentacion(self):
        ...


class Perro(AnimalRefugio):
    ESPECIE = "Perro"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Guau!"

    def tipo_alimentacion(self):
        return "omnívoro"


class Gato(AnimalRefugio):
    ESPECIE = "Gato"

    def hacer_sonido(self):
        return f"{self.nombre} dice: ¡Miau!"

    def tipo_alimentacion(self):
        return "carnívoro"


CLASES = {"Perro": Perro, "Gato": Gato}


def registro_a_animal(registro):
    clase = CLASES.get(registro["especie"])
    if clase is None:
        raise ValueError(f"Demo POO: sin clase para {registro['especie']}")
    return clase(
        registro["nombre"],
        registro["edad"],
        registro["peso"],
        registro["color"],
        registro.get("observaciones", ""),
    )


# Demostración rápida (polimorfismo + property + decorador)
demo = registro_a_animal(animales[1])
print(demo.info_basica())
print(demo.hacer_sonido(), "|", demo.tipo_alimentacion())
print("Usos de info_basica:", demo._usos_info_basica)
try:
    demo.peso = 150  # fuera de rango → ValueError del setter
except ValueError as error:
    print("Setter peso:", error)


## 3. Prueba rápida de las cuatro operaciones

Diccionario vacío independiente para no alterar los registros iniciales.


In [ ]:
inventario_prueba = {}
datos_prueba = {
    "nombre": "Nube",
    "especie": "Gato",
    "edad": 2,
    "peso": 3.8,
    "color": "Gris",
    "observaciones": "Ejemplo temporal",
}

id_prueba = crear_animal(datos_prueba, inventario_prueba)  # Crear
print("Crear:", id_prueba, inventario_prueba)
print("Leer:", leer_animal(id_prueba, inventario_prueba))

datos_prueba["edad"] = 3
actualizar_animal(id_prueba, datos_prueba, inventario_prueba)  # Actualizar
print("Actualizar:", leer_animal(id_prueba, inventario_prueba))

borrar_animal(id_prueba, inventario_prueba)  # Borrar
print("Borrar: quedan", len(inventario_prueba), "registros.")

# La validación de peso también rechaza valores fuera de rango:
try:
    validar_datos({**datos_prueba, "peso": 150})
except ValueError as error:
    print("Peso inválido:", error)


## 4. Interfaz gráfica

Versión compacta de la GUI (CRUD + filtros + `Treeview`). En `tarea1/` la
ventana añade ficha clínica, foto, pegado de adjuntos y exportación PDF.

Al seleccionar una fila se muestra un resumen; si la especie tiene clase POO
en la demo (`Perro`/`Gato`), se incluye el sonido polimórfico.


In [ ]:
class GestorAnimales:
    """Interfaz Tkinter para consultar y gestionar el inventario."""

    def __init__(self, raiz, inventario):
        self.raiz = raiz
        self.inventario = inventario
        self.nombre_var = tk.StringVar()
        self.especie_var = tk.StringVar(value=ESPECIES[0])
        self.edad_var = tk.StringVar()
        self.peso_var = tk.StringVar()
        self.color_var = tk.StringVar()
        self.busqueda_var = tk.StringVar()
        self.filtro_var = tk.StringVar(value="Todas")
        self.estado_var = tk.StringVar(
            value="Selecciona un animal o completa el formulario."
        )

        self.raiz.title("Refugio Animal - Gestión de animales (demo cuaderno)")
        self.raiz.geometry("980x620")
        self.raiz.minsize(820, 560)
        self._crear_controles()
        self._actualizar_tabla()

    def _crear_controles(self):
        contenedor = ttk.Frame(self.raiz, padding=12)
        contenedor.pack(fill="both", expand=True)

        ttk.Label(contenedor, text="Refugio Animal").pack(anchor="w")
        ttk.Label(
            contenedor,
            text="Demo del cuaderno — el programa completo está en tarea1/",
        ).pack(anchor="w", pady=(0, 10))

        formulario = ttk.LabelFrame(contenedor, text="Datos del animal", padding=10)
        formulario.pack(fill="x", pady=(0, 8))
        formulario.columnconfigure(1, weight=1)
        formulario.columnconfigure(3, weight=1)

        ttk.Label(formulario, text="Nombre:").grid(row=0, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.nombre_var).grid(
            row=0, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Especie:").grid(row=0, column=2, sticky="w", padx=5, pady=4)
        ttk.Combobox(
            formulario,
            textvariable=self.especie_var,
            values=ESPECIES,
            state="readonly",
        ).grid(row=0, column=3, sticky="ew", padx=5, pady=4)

        ttk.Label(formulario, text="Edad (0-30):").grid(row=1, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.edad_var).grid(
            row=1, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Peso (0.01-100 kg):").grid(
            row=1, column=2, sticky="w", padx=5, pady=4
        )
        ttk.Entry(formulario, textvariable=self.peso_var).grid(
            row=1, column=3, sticky="ew", padx=5, pady=4
        )

        ttk.Label(formulario, text="Color:").grid(row=2, column=0, sticky="w", padx=5, pady=4)
        ttk.Entry(formulario, textvariable=self.color_var).grid(
            row=2, column=1, sticky="ew", padx=5, pady=4
        )
        ttk.Label(formulario, text="Observaciones:").grid(
            row=2, column=2, sticky="nw", padx=5, pady=4
        )
        self.observaciones_text = tk.Text(formulario, height=3, width=35, wrap="word")
        self.observaciones_text.grid(row=2, column=3, sticky="ew", padx=5, pady=4)

        botones = ttk.Frame(formulario)
        botones.grid(row=3, column=0, columnspan=4, sticky="w", pady=(8, 0))
        ttk.Button(botones, text="Añadir", command=self._crear).pack(side="left", padx=3)
        ttk.Button(botones, text="Actualizar", command=self._actualizar).pack(side="left", padx=3)
        ttk.Button(botones, text="Borrar", command=self._borrar).pack(side="left", padx=3)
        ttk.Button(botones, text="Limpiar", command=self._limpiar).pack(side="left", padx=3)

        filtros = ttk.LabelFrame(contenedor, text="Buscar y filtrar", padding=8)
        filtros.pack(fill="x", pady=(0, 8))
        ttk.Label(filtros, text="Nombre:").pack(side="left", padx=(0, 5))
        busqueda = ttk.Entry(filtros, textvariable=self.busqueda_var, width=28)
        busqueda.pack(side="left", padx=(0, 14))
        busqueda.bind("<KeyRelease>", lambda evento: self._actualizar_tabla())
        ttk.Label(filtros, text="Especie:").pack(side="left", padx=(0, 5))
        filtro = ttk.Combobox(
            filtros,
            textvariable=self.filtro_var,
            values=("Todas",) + ESPECIES,
            state="readonly",
            width=14,
        )
        filtro.pack(side="left")
        filtro.bind("<<ComboboxSelected>>", lambda evento: self._actualizar_tabla())

        tabla_marco = ttk.Frame(contenedor)
        tabla_marco.pack(fill="both", expand=True)
        columnas = ("id", "nombre", "especie", "edad", "peso", "color", "observaciones")
        self.tabla = ttk.Treeview(
            tabla_marco,
            columns=columnas,
            show="headings",
            selectmode="browse",
            height=10,
        )
        encabezados = {
            "id": "ID",
            "nombre": "Nombre",
            "especie": "Especie",
            "edad": "Edad",
            "peso": "Peso (kg)",
            "color": "Color",
            "observaciones": "Observaciones",
        }
        anchos = {
            "id": 45,
            "nombre": 110,
            "especie": 85,
            "edad": 55,
            "peso": 75,
            "color": 100,
            "observaciones": 300,
        }
        for columna in columnas:
            self.tabla.heading(columna, text=encabezados[columna])
            self.tabla.column(columna, width=anchos[columna], anchor="w")
        barra_vertical = ttk.Scrollbar(tabla_marco, orient="vertical", command=self.tabla.yview)
        barra_horizontal = ttk.Scrollbar(
            tabla_marco, orient="horizontal", command=self.tabla.xview
        )
        self.tabla.configure(
            yscrollcommand=barra_vertical.set,
            xscrollcommand=barra_horizontal.set,
        )
        self.tabla.grid(row=0, column=0, sticky="nsew")
        barra_vertical.grid(row=0, column=1, sticky="ns")
        barra_horizontal.grid(row=1, column=0, sticky="ew")
        tabla_marco.rowconfigure(0, weight=1)
        tabla_marco.columnconfigure(0, weight=1)
        self.tabla.bind("<<TreeviewSelect>>", self._seleccionar)

        ttk.Label(contenedor, textvariable=self.estado_var).pack(anchor="w", pady=(8, 0))

    def _datos_del_formulario(self):
        try:
            edad = int(self.edad_var.get().strip())
        except ValueError as error:
            raise ValueError("La edad debe escribirse como un número entero.") from error
        try:
            peso = float(self.peso_var.get().strip().replace(",", "."))
        except ValueError as error:
            raise ValueError(
                "El peso debe escribirse como un número, por ejemplo 4.5."
            ) from error
        return {
            "nombre": self.nombre_var.get(),
            "especie": self.especie_var.get(),
            "edad": edad,
            "peso": peso,
            "color": self.color_var.get(),
            "observaciones": self.observaciones_text.get("1.0", "end"),
        }

    def _id_seleccionado(self):
        seleccion = self.tabla.selection()
        if not seleccion:
            messagebox.showwarning(
                "Falta seleccionar", "Selecciona un animal de la tabla."
            )
            return None
        return int(seleccion[0])

    def _crear(self):
        try:
            animal_id = crear_animal(self._datos_del_formulario(), self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo crear el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Animal creado con el identificador {animal_id}.")

    def _actualizar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        try:
            actualizar_animal(
                animal_id, self._datos_del_formulario(), self.inventario
            )
        except ValueError as error:
            messagebox.showerror("No se pudo actualizar el registro", str(error))
            return
        self._actualizar_tabla()
        self.estado_var.set(f"Registro {animal_id} actualizado.")

    def _borrar(self):
        animal_id = self._id_seleccionado()
        if animal_id is None:
            return
        nombre = self.inventario[animal_id]["nombre"]
        if not messagebox.askyesno("Confirmar borrado", f"¿Quieres borrar a {nombre}?"):
            return
        try:
            borrar_animal(animal_id, self.inventario)
        except ValueError as error:
            messagebox.showerror("No se pudo borrar el registro", str(error))
            return
        self._actualizar_tabla()
        self._limpiar()
        self.estado_var.set(f"Registro {animal_id} eliminado.")

    def _seleccionar(self, evento=None):
        seleccion = self.tabla.selection()
        if not seleccion:
            return
        animal_id = int(seleccion[0])
        registro = leer_animal(animal_id, self.inventario)
        self.nombre_var.set(registro["nombre"])
        self.especie_var.set(registro["especie"])
        self.edad_var.set(str(registro["edad"]))
        self.peso_var.set(str(registro["peso"]))
        self.color_var.set(registro["color"])
        self.observaciones_text.delete("1.0", "end")
        self.observaciones_text.insert("1.0", registro["observaciones"])
        try:
            animal = registro_a_animal(registro)
            extra = f" | {animal.hacer_sonido()} | {animal.tipo_alimentacion()}"
        except ValueError:
            extra = " | (capa POO completa en tarea1 para todas las especies)"
        self.estado_var.set(f"Registro {animal_id} seleccionado{extra}")

    def _limpiar(self):
        self.nombre_var.set("")
        self.especie_var.set(ESPECIES[0])
        self.edad_var.set("")
        self.peso_var.set("")
        self.color_var.set("")
        self.observaciones_text.delete("1.0", "end")
        seleccion = self.tabla.selection()
        if seleccion:
            self.tabla.selection_remove(seleccion)

    def _actualizar_tabla(self):
        for fila in self.tabla.get_children():
            self.tabla.delete(fila)

        texto = self.busqueda_var.get().strip().casefold()
        especie = self.filtro_var.get()
        visibles = 0
        for animal_id, registro in self.inventario.items():
            coincide_nombre = texto in registro["nombre"].casefold()
            coincide_especie = especie == "Todas" or registro["especie"] == especie
            if coincide_nombre and coincide_especie:
                self.tabla.insert(
                    "",
                    "end",
                    iid=str(animal_id),
                    values=(
                        animal_id,
                        registro["nombre"],
                        registro["especie"],
                        registro["edad"],
                        f"{registro['peso']:.2f}",
                        registro["color"],
                        registro["observaciones"],
                    ),
                )
                visibles += 1
        self.estado_var.set(
            f"Registros totales: {len(self.inventario)} | visibles: {visibles}"
        )


## 5. Ejecutar la aplicación (demo del cuaderno)

Orden: **Reiniciar kernel → Ejecutar todas**. La ventana se abre en esta celda.

Para la entrega / defensa con ficha clínica, PDF y tests:

```bash
cd tarea1
python app.py
python ejecutar_tests.py
```


In [ ]:
raiz = tk.Tk()
aplicacion = GestorAnimales(raiz, animales)
raiz.mainloop()
